In [20]:
import numpy as np
import matplotlib.pyplot as plt
from common import softmax, cross_entropy_error, numerical_gradient

In [21]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y

        return out

    def backward(self, dout):
        dx = dout * self.y  # x <-> y 교체
        dy = dout * self.x

        return dx, dy

In [22]:
apple = 100    # 가격
apple_num = 2  # 수량
tax = 1.1      # 소비세

# 계층들
mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer.forward(apple, apple_num)  # 사과 가격 계산
price = mul_tax_layer.forward(apple_price, tax)          # 소비세 가격 계산

# 역전파
dprice = 1   # 가격이 1 일때
dapple_price, dtax = mul_tax_layer.backward(dprice)          # 소비세 계층 역전파
dapple, dapple_num = mul_apple_layer.backward(dapple_price)  # 사과 계층 역전파

print(dapple, dapple_num, dtax)  # 2.2 100 200 
# 의미 : 가격 1당 사과 가격의 변화량, 사과 수량의 변화량, 소비세의 변화량

2.2 110.00000000000001 200


In [23]:
class AddLayer:
    def __init__(self): # 초기화 필요 X
        pass

    def forward(self, x, y):
        out = x + y
        return out

    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        return dx, dy

In [24]:
apple = 100   
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# 계층들
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# 순전파
apple_price = mul_apple_layer.forward(apple, apple_num)                # 1
orange_price = mul_orange_layer.forward(orange, orange_num)            # 2
all_price = add_apple_orange_layer.forward(apple_price, orange_price)  # 3
price = mul_tax_layer.forward(all_price, tax)                          # 4

# 역전파
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)                          # 4
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)  # 3
dorange, dorange_num = mul_orange_layer.backward(dorange_price)            # 2
dapple, dapple_num = mul_apple_layer.backward(dapple_price)                #1

print(price)  
print(dapple_num, dapple, dorange, dorange_num, dtax)

715.0000000000001
110.00000000000001 2.2 3.3000000000000003 165.0 650


In [25]:
class Relu:
    
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

In [26]:
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
print(x)
mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


In [27]:
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = 1 / (1 + np.exp(-x))
        self.out = out

        return out

    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out
        
        return dx

In [28]:
X_dot_W = np.array([[0, 0, 0], [10, 10, 10]])
B = np.array([1, 2, 3])

print(X_dot_W + B)

dY = np.array([[1, 2, 3], [4, 5, 6]])
dB = np.sum(dY, axis=0)
print(dB) 

[[ 1  2  3]
 [11 12 13]]
[5 7 9]


In [29]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(self.x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)

        return dx

In [30]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None  # softmax의 출력
        self.t = None  # 정답 레이블(one-hot vector)

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)

        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size

        return dx

In [31]:
import sys, os
from collections import OrderedDict

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # 계층 생성
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    # x : 입력 데이터, t : 정답 레이블
    def loss(self, x, t):
        y = self.predict(x)

        return self.last_layer.forward(y, t)


    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)
        accuracy = np.sum(y == t) / float(x.shape[0])

        return accuracy

    # x : 입력 데이터, t : 정답 레이블
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

    def gradient(self, x, t):
        # 순전파
        self.loss(x, t)

        # 역전파
        dout = 1
        dout = self.last_layer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 결과 저장
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads

In [33]:
from mnist import load_mnist

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grads_numerical = network.numerical_gradient(x_batch, t_batch)
grads_backprop = network.gradient(x_batch, t_batch)

# 각 가중치의 차이의 절댓값을 구한 후, 그 절댓값들의 평균을 낸다.
for key in grads_numerical.keys():
    diff = np.average(np.abs(grads_backprop[key] - grads_numerical[key]))
    print(key + ":" + str(diff))

W1:4.1119650979463566e-10
b1:2.4084456633804617e-09
W2:6.0852415301516395e-09
b2:1.4004507319087534e-07


In [34]:
# 데이터 읽기 
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 기울기 계산 -> 오차역전파법 사용
    grad = network.gradient(x_batch, t_batch)

    # 매개변수 갱신
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc))

train acc, test acc | 0.14733333333333334, 0.1401
train acc, test acc | 0.9018333333333334, 0.9063
train acc, test acc | 0.9207833333333333, 0.9256
train acc, test acc | 0.9354833333333333, 0.9364
train acc, test acc | 0.94455, 0.9437
train acc, test acc | 0.9509666666666666, 0.9497
train acc, test acc | 0.9559333333333333, 0.9516
train acc, test acc | 0.95735, 0.9535
train acc, test acc | 0.9627166666666667, 0.957
train acc, test acc | 0.9660833333333333, 0.96
train acc, test acc | 0.96865, 0.962
train acc, test acc | 0.97035, 0.9627
train acc, test acc | 0.9713833333333334, 0.9639
train acc, test acc | 0.9735166666666667, 0.9641
train acc, test acc | 0.97485, 0.9653
train acc, test acc | 0.9762666666666666, 0.9678
train acc, test acc | 0.97715, 0.9661
